# RACE Project — Preprocessing Pipeline
**Course:** Artificial Intelligence — BS(CS) Spring 2026  
**Dataset:** RACE (ReAding Comprehension from Examinations)  

**Inputs:** `data/processed/train_split.csv`, `data/processed/val_split.csv`  
**Outputs:** TF-IDF matrices, OHE matrices, cosine features, lexical features, combined matrix, labels  

---
### What this notebook produces
1. Setup & folder creation  
2. Load train/val splits  
3. Text cleaning  
4. Label encoding  
5. Build verification texts (article×2 + question + option)  
6. TF-IDF features  
7. One-Hot Encoding features  
8. Cosine similarity features  
9. Lexical features  
10. Build combined feature matrix  
11. Final verification

## Cell 1 — Setup & Imports

In [29]:
import os
import re
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)

# Create all folders the project needs
for path in [
    '../data/processed',
    'models/model_a/traditional',
    'models/model_b/traditional',
]:
    os.makedirs(path, exist_ok=True)

print('Setup complete.')

Setup complete.


## Cell 2 — Load Train / Val Splits

> **Important:** Load `train_split.csv` and `val_split.csv` — NOT the original `train.csv`.  
> The splits were created with stratified 80/20 sampling in the EDA notebook.

In [30]:
train_df = pd.read_csv('../data/processed/train_split.csv')
val_df   = pd.read_csv('../data/processed/val_split.csv')

print(f'Train : {train_df.shape}')
print(f'Val   : {val_df.shape}')
print(f'Columns: {train_df.columns.tolist()}')
print(f'\nAnswer distribution (train):')
print(train_df['answer'].value_counts().sort_index())
print(f'\nAnswer distribution (val):')
print(val_df['answer'].value_counts().sort_index())

Train : (70281, 15)
Val   : (17571, 15)
Columns: ['id', 'article', 'question', 'A', 'B', 'C', 'D', 'answer', 'article_len', 'question_len', 'A_len', 'B_len', 'C_len', 'D_len', 'question_type']

Answer distribution (train):
answer
A    15314
B    18177
C    19109
D    17681
Name: count, dtype: int64

Answer distribution (val):
answer
A    3828
B    4545
C    4778
D    4420
Name: count, dtype: int64


## Cell 3 — Text Cleaning

Cleaning steps:
1. Cast to string (handles accidental floats)
2. Lowercase
3. Remove URLs
4. Remove punctuation
5. Remove standalone numbers
6. Collapse whitespace

> **Note:** Articles are truncated at 500 words (EDA 95th pct = ~480 words).  
> Questions and options are **never** truncated — they are already short.

In [31]:
def clean_text(text):
    """
    Clean a single piece of text.
    Steps:
      1. Convert to string
      2. Lowercase everything
      3. Remove URLs
      4. Remove punctuation
      5. Remove standalone numbers
      6. Remove extra whitespace
    """
    text = str(text)                               # handle accidental floats
    text = text.lower()                            # STEP 2: lowercase
    text = re.sub(r'http\S+|www\.\S+', '', text)   # STEP 3: remove URLs
    text = re.sub(r'[^\w\s]', ' ', text)           # STEP 4: remove punctuation
    text = re.sub(r'\b\d+\b', ' ', text)           # STEP 5: remove standalone numbers
    text = re.sub(r'\s+', ' ', text).strip()       # STEP 6: clean whitespace
    return text


def truncate_article(text, max_words=500):
    """
    Truncate article to max_words AFTER cleaning.
    EDA: 95th percentile = ~480 words, so 500 keeps 95% of articles fully intact.
    Questions and options are never truncated.
    """
    words = text.split()
    if len(words) > max_words:
        return ' '.join(words[:max_words])
    return text


def clean_dataframe(df):
    """Apply clean_text to all 6 text columns, then truncate articles."""
    df = df.copy()
    for col in ['article', 'question', 'A', 'B', 'C', 'D']:
        df[f'{col}_clean'] = df[col].apply(clean_text)
    # Truncate articles — keeps 95% of articles fully intact
    # Questions and options are already short — never truncate
    df['article_clean'] = df['article_clean'].apply(truncate_article)
    return df


# Apply to both splits
print('Cleaning train...')
train_df = clean_dataframe(train_df)

print('Cleaning val...')
val_df   = clean_dataframe(val_df)

# Spot check — verify cleaning worked
print('\nBefore:', repr(train_df['article'].iloc[0][:80]))
print('After :', repr(train_df['article_clean'].iloc[0][:80]))
print(f'\nArticle length after truncation (max): {train_df["article_clean"].str.split().str.len().max()} words')
print('Cleaning complete.')

Cleaning train...
Cleaning val...

Before: 'Giant sunflowers? Maybe you like potatoes, tomatoes, or carrots.Or maybe you pre'
After : 'giant sunflowers maybe you like potatoes tomatoes or carrots or maybe you prefer'

Article length after truncation (max): 500 words
Cleaning complete.


## Cell 4 — Label Encoding

Map `A/B/C/D` → `0/1/2/3`.  
EDA confirmed ~25% per class — do NOT use `class_weight='balanced'`.

In [32]:
# Map letters to integers
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

y_train = train_df['answer'].map(label_map).values
y_val   = val_df['answer'].map(label_map).values

print(f'y_train shape : {y_train.shape}')
print(f'y_val shape   : {y_val.shape}')
print(f'Classes       : {np.unique(y_train)}  (0=A, 1=B, 2=C, 3=D)')
print(f'Class balance : {np.bincount(y_train) / len(y_train)}')

# Save immediately
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy',   y_val)
print('Labels saved.')

y_train shape : (70281,)
y_val shape   : (17571,)
Classes       : [0 1 2 3]  (0=A, 1=B, 2=C, 3=D)
Class balance : [0.21789673 0.2586332  0.27189425 0.25157582]
Labels saved.


## Cell 5 — Build Verification Texts

Each row expands to **4** candidate strings (one per option A/B/C/D).  
Format: `article + article + question + option`

**Why article twice?** EDA shows articles average ~245 words, options average ~5.7 words.  
Repeating the article gives it proportional weight against the short option in TF-IDF space.

Binary labels: 1 = this option is the correct answer, 0 = distractor.

In [33]:
def build_verification_texts(df):
    """
    For each row create 4 combined strings (one per option A/B/C/D).
    Format: article + article + question + option
    Article is repeated twice to give it more weight than the short option.

    Returns:
        texts  : list of strings  (4 × number of rows)
        labels : list of 0 or 1   (1 = this option is correct)
    """
    texts  = []
    labels = []

    for _, row in df.iterrows():
        art     = row['article_clean']
        q       = row['question_clean']
        correct = row['answer']           # 'A', 'B', 'C', or 'D'

        for opt in ['A', 'B', 'C', 'D']:
            opt_text = row[f'{opt}_clean']
            combined = f'{art} {art} {q} {opt_text}'
            texts.append(combined)
            labels.append(1 if opt == correct else 0)

    return texts, labels


print('Building verification texts for train...')
train_texts, train_bin_labels = build_verification_texts(train_df)

print('Building verification texts for val...')
val_texts,   val_bin_labels   = build_verification_texts(val_df)

print(f'\nTrain verification samples : {len(train_texts):,}')
print(f'Expected (train rows × 4)  : {len(train_df)*4:,}')
print(f'Positive rate              : {np.mean(train_bin_labels):.3f} (should be ~0.25)')

# Save binary labels
np.save('../data/processed/y_train_binary.npy', np.array(train_bin_labels))
np.save('../data/processed/y_val_binary.npy',   np.array(val_bin_labels))
print('Binary labels saved.')

Building verification texts for train...
Building verification texts for val...

Train verification samples : 281,124
Expected (train rows × 4)  : 281,124
Positive rate              : 0.250 (should be ~0.25)
Binary labels saved.


## Cell 6 — TF-IDF Features

Settings justified by EDA:
- `max_features=10000` → covers 89% of all content tokens  
- `min_df=2` → removes ~43,000 single-occurrence words (typos, OCR artifacts)  
- `max_df=0.95` → removes near-universal terms  
- `sublinear_tf=True` → dampens very frequent terms  
- `ngram_range=(1,2)` → unigrams + bigrams  
- `norm='l2'` → **required** for cosine similarity to work correctly

> **Critical:** `fit_transform()` on TRAIN only. `transform()` on VAL.  
> Never refit on val/test — causes data leakage and inflated metrics.

In [34]:
print('Fitting TF-IDF vectorizer on train...')

tfidf_vectorizer = TfidfVectorizer(
    max_features = 10000,      # justified by vocabulary coverage analysis (89% of content)
    stop_words   = 'english',  # justified by raw word frequency analysis
    sublinear_tf = True,       # dampens very frequent terms
    ngram_range  = (1, 2),     # unigrams + bigrams
    min_df       = 2,          # removes ~43,000 single-occurrence words
    max_df       = 0.95,       # removes near-universal terms
    norm         = 'l2',       # required for cosine similarity to work
)

# FIT on train ONLY — NEVER refit on val/test
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)

# TRANSFORM val — no fitting
X_val_tfidf   = tfidf_vectorizer.transform(val_texts)

print(f'Train TF-IDF shape : {X_train_tfidf.shape}')
print(f'Val TF-IDF shape   : {X_val_tfidf.shape}')
print(f'Sparsity           : {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.1%}')

# Always use scipy sparse — never .toarray() on the full matrix (~6.5 GB dense)
sp.save_npz('../data/processed/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('../data/processed/X_val_tfidf.npz',   X_val_tfidf)

# Save vectorizer — model is useless at inference time without it
joblib.dump(tfidf_vectorizer, 'models/model_a/traditional/tfidf_vectorizer.pkl')
joblib.dump(tfidf_vectorizer, 'models/model_b/traditional/tfidf_vectorizer.pkl')

print('TF-IDF matrices and vectorizer saved.')

Fitting TF-IDF vectorizer on train...
Train TF-IDF shape : (281124, 10000)
Val TF-IDF shape   : (70284, 10000)
Sparsity           : 99.0%
TF-IDF matrices and vectorizer saved.


## Cell 7 — One-Hot Encoding Features

`binary=True` in `CountVectorizer` makes it a true bag-of-words presence matrix (1 = word present, 0 = absent).  
This is separate from TF-IDF and captures different signal: occurrence vs. frequency.

In [35]:
print('Fitting One-Hot Encoding vectorizer on train...')

ohe_vectorizer = CountVectorizer(
    max_features = 8000,       # slightly smaller — OHE is already noisier than TF-IDF
    stop_words   = 'english',
    binary       = True,       # THIS makes it One-Hot (presence, not count)
    ngram_range  = (1, 1),     # unigrams only for OHE
    min_df       = 2,
    max_df       = 0.95,
)

# FIT on train ONLY
X_train_ohe = ohe_vectorizer.fit_transform(train_texts)

# TRANSFORM val
X_val_ohe   = ohe_vectorizer.transform(val_texts)

print(f'Train OHE shape : {X_train_ohe.shape}')
print(f'Val OHE shape   : {X_val_ohe.shape}')

# Save
sp.save_npz('../data/processed/X_train_ohe.npz', X_train_ohe)
sp.save_npz('../data/processed/X_val_ohe.npz',   X_val_ohe)
joblib.dump(ohe_vectorizer, 'models/model_a/traditional/ohe_vectorizer.pkl')
joblib.dump(ohe_vectorizer, 'models/model_b/traditional/ohe_vectorizer.pkl')

print('OHE matrices and vectorizer saved.')

Fitting One-Hot Encoding vectorizer on train...
Train OHE shape : (281124, 8000)
Val OHE shape   : (70284, 8000)
OHE matrices and vectorizer saved.


## Cell 8 — Cosine Similarity Features

**Motivation from EDA:**
- Mean answer–article overlap = 0.670
- 90.1% of questions have > 0% overlap with the correct article passage
- → TF-IDF cosine similarity is a **strong and reliable feature** for this dataset

**Strategy:**  
Fit a *separate* article-only TF-IDF vectorizer. For each of the 4 options per row, compute:
1. `cos(article, option)` — how much the option overlaps the article
2. `cos(article, question)` — how much the question overlaps the article
3. `cos(question, option)` — how much the option overlaps the question
4. `cos(article, question+option)` — article vs. combined query+answer
5. `rank_in_group` — rank of this option's cosine score among its 4 siblings (0–3)
6. `is_max_in_group` — 1 if this option has the highest cosine score in its group

> **Critical:** Fit the article vectorizer on **train articles only**. Transform val articles separately.
> This vectorizer uses `norm='l2'` so cosine_similarity = dot product on normalized vectors.

In [36]:
print('Building cosine similarity features...')

# --- 1. Fit article-level TF-IDF on train articles only ---
cos_vectorizer = TfidfVectorizer(
    max_features = 10000,
    stop_words   = 'english',
    sublinear_tf = True,
    ngram_range  = (1, 1),   # unigrams only for overlap comparison
    min_df       = 2,
    max_df       = 0.95,
    norm         = 'l2',     # required — cosine = dot product on l2-normalised vectors
)

# Fit on train articles only — no leakage
cos_vectorizer.fit(train_df['article_clean'])
joblib.dump(cos_vectorizer, 'models/model_a/traditional/cos_vectorizer.pkl')
joblib.dump(cos_vectorizer, 'models/model_b/traditional/cos_vectorizer.pkl')
print('Cosine vectorizer fitted on train articles.')


def build_cosine_features(df, vectorizer):
    """
    Build 6 cosine-similarity features per (row, option) pair.
    Output shape: (n_rows × 4, 6)

    Features:s
        0 : cos(article, option)
        1 : cos(article, question)
        2 : cos(question, option)
        3 : cos(article, question+option)
        4 : rank of this option's cos(art,opt) among its 4 siblings (0=worst, 3=best)
        5 : is_max flag — 1 if this option has the highest cos(art,opt) in its group
    """
    n_rows   = len(df)
    features = np.zeros((n_rows * 4, 6), dtype=np.float32)

    for i, (_, row) in enumerate(df.iterrows()):
        art = row['article_clean']
        q   = row['question_clean']

        # Vectorise article and question once per row
        art_vec = vectorizer.transform([art])
        q_vec   = vectorizer.transform([q])

        cos_art_opt_scores = []

        for j, opt_letter in enumerate(['A', 'B', 'C', 'D']):
            opt_text = row[f'{opt_letter}_clean']
            qopt     = q + ' ' + opt_text          # combined query+answer text

            opt_vec  = vectorizer.transform([opt_text])
            qopt_vec = vectorizer.transform([qopt])

            cos_ao   = cosine_similarity(art_vec, opt_vec)[0, 0]   # art vs option
            cos_aq   = cosine_similarity(art_vec, q_vec)[0, 0]     # art vs question
            cos_qo   = cosine_similarity(q_vec,   opt_vec)[0, 0]   # question vs option
            cos_aqo  = cosine_similarity(art_vec, qopt_vec)[0, 0]  # art vs q+option

            idx = i * 4 + j
            features[idx, 0] = cos_ao
            features[idx, 1] = cos_aq
            features[idx, 2] = cos_qo
            features[idx, 3] = cos_aqo

            cos_art_opt_scores.append(cos_ao)

        # Rank and max flag within this group of 4
        scores  = np.array(cos_art_opt_scores)
        ranks   = scores.argsort().argsort()   # rank 0=worst, 3=best
        is_max  = (scores == scores.max()).astype(np.float32)

        for j in range(4):
            idx = i * 4 + j
            features[idx, 4] = ranks[j]
            features[idx, 5] = is_max[j]

        if (i + 1) % 5000 == 0:
            print(f'  Processed {i+1:,} / {n_rows:,} rows...')

    return features


print('\nComputing cosine features for train...')
cos_train = build_cosine_features(train_df, cos_vectorizer)
print(f'cos_train shape : {cos_train.shape}  (expected: {len(train_df)*4} × 6)')

print('\nComputing cosine features for val...')
cos_val = build_cosine_features(val_df, cos_vectorizer)
print(f'cos_val shape   : {cos_val.shape}  (expected: {len(val_df)*4} × 6)')

# Save
np.save('../data/processed/cosine_features_train.npy', cos_train)
np.save('../data/processed/cosine_features_val.npy',   cos_val)
print('Cosine features saved.')

Building cosine similarity features...
Cosine vectorizer fitted on train articles.

Computing cosine features for train...
  Processed 5,000 / 70,281 rows...
  Processed 10,000 / 70,281 rows...
  Processed 15,000 / 70,281 rows...
  Processed 20,000 / 70,281 rows...
  Processed 25,000 / 70,281 rows...
  Processed 30,000 / 70,281 rows...
  Processed 35,000 / 70,281 rows...
  Processed 40,000 / 70,281 rows...
  Processed 45,000 / 70,281 rows...
  Processed 50,000 / 70,281 rows...
  Processed 55,000 / 70,281 rows...
  Processed 60,000 / 70,281 rows...
  Processed 65,000 / 70,281 rows...
  Processed 70,000 / 70,281 rows...
cos_train shape : (281124, 6)  (expected: 281124 × 6)

Computing cosine features for val...
  Processed 5,000 / 17,571 rows...
  Processed 10,000 / 17,571 rows...
  Processed 15,000 / 17,571 rows...
cos_val shape   : (70284, 6)  (expected: 70284 × 6)
Cosine features saved.


## Cell 9 — Lexical Features

**Motivation from EDA:**
- Mean answer–article word overlap = 0.670 → high overlap dataset  
- Distractor avg Jaccard vs. correct answer = ~0.05 → distractors use different words  
- → Simple word-matching features add signal beyond TF-IDF

**13 features per (row, option) pair:**

| # | Feature | Notes |
|---|---------|-------|
| 0 | word_overlap_art_opt | # words shared between article and option |
| 1 | jaccard_art_opt | Jaccard similarity: art ∩ opt / art ∪ opt |
| 2 | overlap_ratio_opt | overlap / len(option words) |
| 3 | overlap_ratio_art | overlap / len(article words) |
| 4 | word_overlap_q_opt | # words shared between question and option |
| 5 | jaccard_q_opt | Jaccard: question ∩ option / question ∪ option |
| 6 | art_len | article word count (after truncation) |
| 7 | opt_len | option word count |
| 8 | q_len | question word count |
| 9 | len_ratio | opt_len / (art_len + 1) |
| 10 | rank_in_group | rank of word_overlap_art_opt among 4 options (0=worst, 3=best) |
| 11 | is_max_in_group | 1 if this option has highest overlap in its group |
| 12 | question_type_encoded | 0=Fill-in-blank, 1=Wh-question, 2=Other |

In [37]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOPWORDS = set(ENGLISH_STOP_WORDS)


def get_words(text):
    """Return set of non-stopword tokens from cleaned text."""
    return set(text.split()) - STOPWORDS


def get_question_type_code(question):
    """
    EDA Section 6: question types matter for Model A template selection.
    Encode as integer for use as a feature.
      0 = Fill-in-blank (contains underscore or 'blank')
      1 = Wh-question (what/who/where/when/why/how/which)
      2 = Other
    """
    q = str(question).lower().strip()
    if '_' in q or 'blank' in q:
        return 0
    for wh in ['what', 'who', 'where', 'when', 'why', 'how', 'which']:
        if q.startswith(wh):
            return 1
    return 2


def build_lexical_features(df):
    """
    Build 13 lexical features per (row, option) pair.
    Output shape: (n_rows × 4, 13)
    """
    n_rows   = len(df)
    features = np.zeros((n_rows * 4, 13), dtype=np.float32)

    for i, (_, row) in enumerate(df.iterrows()):
        art_words = get_words(row['article_clean'])
        q_words   = get_words(row['question_clean'])
        art_len   = len(row['article_clean'].split())
        q_len     = len(row['question_clean'].split())
        qtype     = get_question_type_code(row['question_clean'])

        overlap_scores = []

        for j, opt_letter in enumerate(['A', 'B', 'C', 'D']):
            opt_text  = row[f'{opt_letter}_clean']
            opt_words = get_words(opt_text)
            opt_len   = len(opt_text.split())

            # Article vs Option overlap
            inter_ao  = art_words & opt_words
            union_ao  = art_words | opt_words
            n_overlap = len(inter_ao)
            jaccard_ao = n_overlap / len(union_ao) if union_ao else 0.0
            ratio_opt  = n_overlap / (opt_len + 1)
            ratio_art  = n_overlap / (art_len + 1)

            # Question vs Option overlap
            inter_qo   = q_words & opt_words
            union_qo   = q_words | opt_words
            n_q_overlap = len(inter_qo)
            jaccard_qo  = n_q_overlap / len(union_qo) if union_qo else 0.0

            idx = i * 4 + j
            features[idx, 0]  = n_overlap
            features[idx, 1]  = jaccard_ao
            features[idx, 2]  = ratio_opt
            features[idx, 3]  = ratio_art
            features[idx, 4]  = n_q_overlap
            features[idx, 5]  = jaccard_qo
            features[idx, 6]  = art_len
            features[idx, 7]  = opt_len
            features[idx, 8]  = q_len
            features[idx, 9]  = opt_len / (art_len + 1)
            features[idx, 12] = qtype

            overlap_scores.append(n_overlap)

        # Rank and max flag within this group of 4
        scores = np.array(overlap_scores, dtype=np.float32)
        ranks  = scores.argsort().argsort()   # 0=worst, 3=best
        is_max = (scores == scores.max()).astype(np.float32)

        for j in range(4):
            idx = i * 4 + j
            features[idx, 10] = ranks[j]
            features[idx, 11] = is_max[j]

        if (i + 1) % 5000 == 0:
            print(f'  Processed {i+1:,} / {n_rows:,} rows...')

    return features


print('Building lexical features for train...')
lex_train = build_lexical_features(train_df)
print(f'lex_train shape : {lex_train.shape}  (expected: {len(train_df)*4} × 13)')

print('\nBuilding lexical features for val...')
lex_val = build_lexical_features(val_df)
print(f'lex_val shape   : {lex_val.shape}  (expected: {len(val_df)*4} × 13)')

# Save
np.save('../data/processed/lexical_features_train.npy', lex_train)
np.save('../data/processed/lexical_features_val.npy',   lex_val)
print('Lexical features saved.')

# Sanity check — verify mean article-option overlap matches EDA
mean_overlap = lex_train[:, 1].mean()   # feature 1 = jaccard_art_opt
print(f'\nMean Jaccard art-opt (train): {mean_overlap:.3f}  (EDA reported ~0.670 for word overlap rate)')

Building lexical features for train...
  Processed 5,000 / 70,281 rows...
  Processed 10,000 / 70,281 rows...
  Processed 15,000 / 70,281 rows...
  Processed 20,000 / 70,281 rows...
  Processed 25,000 / 70,281 rows...
  Processed 30,000 / 70,281 rows...
  Processed 35,000 / 70,281 rows...
  Processed 40,000 / 70,281 rows...
  Processed 45,000 / 70,281 rows...
  Processed 50,000 / 70,281 rows...
  Processed 55,000 / 70,281 rows...
  Processed 60,000 / 70,281 rows...
  Processed 65,000 / 70,281 rows...
  Processed 70,000 / 70,281 rows...
lex_train shape : (281124, 13)  (expected: 281124 × 13)

Building lexical features for val...
  Processed 5,000 / 17,571 rows...
  Processed 10,000 / 17,571 rows...
  Processed 15,000 / 17,571 rows...
lex_val shape   : (70284, 13)  (expected: 70284 × 13)
Lexical features saved.

Mean Jaccard art-opt (train): 0.021  (EDA reported ~0.670 for word overlap rate)


## Cell 10 — Build Combined Feature Matrix

Stack all feature blocks horizontally:
```
X_combined = [TF-IDF | OHE | cosine_features | lexical_features]
             [10000  | 8000|       6          |       13        ]
             = 18,019 features total
```

> Always use `scipy.sparse` format — the dense version would be ~6.5 GB and crash.

In [38]:
def build_combined(X_tfidf, X_ohe, cos_feats, lex_feats):
    """
    Horizontally stack all feature blocks into one sparse CSR matrix.
    cos_feats and lex_feats are dense numpy arrays — convert to sparse first.
    """
    cos_sparse = sp.csr_matrix(cos_feats)
    lex_sparse = sp.csr_matrix(lex_feats)
    return sp.hstack([X_tfidf, X_ohe, cos_sparse, lex_sparse], format='csr')


print('Building combined matrices...')
X_train_combined = build_combined(X_train_tfidf, X_train_ohe, cos_train, lex_train)
X_val_combined   = build_combined(X_val_tfidf,   X_val_ohe,   cos_val,   lex_val)

print(f'Combined train shape : {X_train_combined.shape}')
print(f'Combined val shape   : {X_val_combined.shape}')
print(f'Expected features    : 10000 + 8000 + 6 + 13 = {10000 + 8000 + 6 + 13}')

# Save — always .npz for sparse matrices
sp.save_npz('../data/processed/X_train_combined.npz', X_train_combined)
sp.save_npz('../data/processed/X_val_combined.npz',   X_val_combined)
print('Combined matrices saved.')

Building combined matrices...
Combined train shape : (281124, 18019)
Combined val shape   : (70284, 18019)
Expected features    : 10000 + 8000 + 6 + 13 = 18019
Combined matrices saved.


## Cell 11 — Final Verification

Check every output file exists and has a nonzero size.

In [ ]:
print('=== FINAL VERIFICATION ===\n')

files = [
    '../data/processed/y_train.npy',
    '../data/processed/y_val.npy',
    '../data/processed/y_train_binary.npy',
    '../data/processed/y_val_binary.npy',
    '../data/processed/X_train_tfidf.npz',
    '../data/processed/X_val_tfidf.npz',
    '../data/processed/X_train_ohe.npz',
    '../data/processed/X_val_ohe.npz',
    '../data/processed/cosine_features_train.npy',
    '../data/processed/cosine_features_val.npy',
    '../data/processed/lexical_features_train.npy',
    '../data/processed/lexical_features_val.npy',
    '../data/processed/X_train_combined.npz',
    '../data/processed/X_val_combined.npz',
    'models/model_a/traditional/tfidf_vectorizer.pkl',
    'models/model_a/traditional/ohe_vectorizer.pkl',
    'models/model_a/traditional/cos_vectorizer.pkl',
    'models/model_b/traditional/tfidf_vectorizer.pkl',
    'models/model_b/traditional/ohe_vectorizer.pkl',
    'models/model_b/traditional/cos_vectorizer.pkl',
]

all_good = True
for f in files:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) / 1024**2 if exists else 0
    status = '✅' if exists else '❌'
    print(f'{status}  {f:60s}  {size:.1f} MB')
    if not exists:
        all_good = False

print()

# Shape sanity check
print('=== SHAPE CHECKS ===\n')
shapes = {
    'X_train_tfidf'    : sp.load_npz('../data/processed/X_train_tfidf.npz').shape,
    'X_val_tfidf'      : sp.load_npz('../data/processed/X_val_tfidf.npz').shape,
    'X_train_ohe'      : sp.load_npz('../data/processed/X_train_ohe.npz').shape,
    'X_val_ohe'        : sp.load_npz('../data/processed/X_val_ohe.npz').shape,
    'X_train_combined' : sp.load_npz('../data/processed/X_train_combined.npz').shape,
    'X_val_combined'   : sp.load_npz('../data/processed/X_val_combined.npz').shape,
    'cos_train'        : np.load('../data/processed/cosine_features_train.npy').shape,
    'cos_val'          : np.load('../data/processed/cosine_features_val.npy').shape,
    'lex_train'        : np.load('../data/processed/lexical_features_train.npy').shape,
    'lex_val'          : np.load('../data/processed/lexical_features_val.npy').shape,
    'y_train'          : np.load('../data/processed/y_train.npy').shape,
    'y_val'            : np.load('../data/processed/y_val.npy').shape,
    'y_train_binary'   : np.load('../data/processed/y_train_binary.npy').shape,
    'y_val_binary'     : np.load('../data/processed/y_val_binary.npy').shape,
}

for name, shape in shapes.items():
    print(f'  {name:25s} : {str(shape):20s}')

print()
if all_good:
    print('✅  All files present — preprocessing complete.')
else:
    print('❌  Some files missing — re-run the cells marked ❌')

=== FINAL VERIFICATION ===

✅  ../data/processed/y_train.npy                                 0.5 MB
✅  ../data/processed/y_val.npy                                   0.1 MB
✅  ../data/processed/y_train_binary.npy                          2.1 MB
✅  ../data/processed/y_val_binary.npy                            0.5 MB
✅  ../data/processed/X_train_tfidf.npz                           201.4 MB
✅  ../data/processed/X_val_tfidf.npz                             50.1 MB
✅  ../data/processed/X_train_ohe.npz                             15.8 MB
✅  ../data/processed/X_val_ohe.npz                               3.7 MB
✅  ../data/processed/cosine_features_train.npy                   6.4 MB
✅  ../data/processed/cosine_features_val.npy                     1.6 MB
✅  ../data/processed/lexical_features_train.npy                  13.9 MB
✅  ../data/processed/lexical_features_val.npy                    3.5 MB
✅  ../data/processed/X_train_combined.npz                        230.4 MB
✅  ../data/processed/X_val_co